> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t08_backfill_bronze_to_silver` · notebook `02_bronze_to_silver` · started 2026-10-11 13:09 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `end_date=2026-10-08`, `mode=backfill`, `start_date=2026-09-25`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 02 · Bronze → Silver
Casts epoch integers to timestamps, decodes titles, removes PII (drops names/avatars, hashes user ids with a
salted SHA-256), derives body features, applies data-quality rules and upserts every Silver table with
**`MERGE INTO`**. Re-running the same batches changes nothing (idempotent).

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: Bronze batches not yet in Silver | `backfill`: re-process the selected batches |
| `batch_ids` | empty | comma-separated batch ids |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `pii_salt` | empty (read from secret `so_lakehouse/pii_salt`) | same |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pii_salt", "")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.silver import run_bronze_to_silver

cfg = build_config(need_salt=True)
audit = run_bronze_to_silver(
    spark, cfg,
    mode=widget("mode"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    fail_on_error=True,
)

Bronze-to-Silver: processing 6 batch(es): full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z
[SUCCESS] Bronze-to-Silver | questions                          | read=7679 ins=0 upd=0 del=0 quar=0 | 29.0s | 7620 unique questions after de-duplication
[SUCCESS] Bronze-to-Silver | question_tags                      | read=24185 ins=0 upd=0 del=0 quar=0 | 13.1s
[SUCCESS] Bronze-to-Silver | answers                            | read=9701 ins=0 upd=0 del=0 quar=0 | 9.4s | 9611 unique answers after de-duplication
[SUCCESS] Bronze-to-Silver | users                              | read=17282 ins=0 upd=0 del=0 quar=0 | 8.1s
[SUCCESS] Bronze-to-Silver | question_snapshots                 | read=7385 ins=0 upd=0 del=0 quar=0 | 4.7s
[SUCCESS] Bronze-to-Silver | question_metrics (from snapshots)  | read=7285 ins=0 upd=0 del=0 quar=0 | 5.9s
[SUCCESS] Bronze-to-Silver | deleted_questions -> questions     | read=

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
bd273545-4804-4f04-b7aa-a20238156e0b,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,questions,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.questions,2026-10-11T13:09:39.959Z,2026-10-11T13:10:08.919Z,28.959547,SUCCESS,7679,0,0,0,0,7620 unique questions after de-duplication,2026-10-11T13:10:08.919Z
842a6ebd-4c31-4f81-8d7b-c9f53450ba08,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,question_tags,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.question_tags,2026-10-11T13:10:12.124Z,2026-10-11T13:10:25.266Z,13.142708,SUCCESS,24185,0,0,0,0,,2026-10-11T13:10:25.266Z
628e4b14-eb79-4241-950b-7e5db73466e1,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,answers,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.answers,2026-10-11T13:10:26.572Z,2026-10-11T13:10:35.947Z,9.374953,SUCCESS,9701,0,0,0,0,9611 unique answers after de-duplication,2026-10-11T13:10:35.947Z
5b2278cc-8382-48e1-9bee-3d21b83f95cf,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,users,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.users,2026-10-11T13:10:37.310Z,2026-10-11T13:10:45.383Z,8.073155,SUCCESS,17282,0,0,0,0,,2026-10-11T13:10:45.383Z
3a2ad0c5-5f79-4d5e-8066-9c06d47bf059,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,question_snapshots,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.question_snapshots,2026-10-11T13:10:46.805Z,2026-10-11T13:10:51.456Z,4.651523,SUCCESS,7385,0,0,0,0,,2026-10-11T13:10:51.456Z
b5dfb835-4046-4439-bcc7-8a33ae1afc00,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,question_metrics (from snapshots),,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.questions,2026-10-11T13:10:52.884Z,2026-10-11T13:10:58.790Z,5.905682,SUCCESS,7285,0,0,0,0,,2026-10-11T13:10:58.790Z
19672975-2454-47b9-bf3d-7b6daee1c29e,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,deleted_questions -> questions,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z, incr_20261007T160033Z, incr_20261008T000000Z]",workspace.so_silver.questions,2026-10-11T13:11:00.086Z,2026-10-11T13:11:05.734Z,5.647287,SUCCESS,1,0,0,0,0,soft delete (is_deleted = true),2026-10-11T13:11:05.734Z
8a60d92f-e933-4e7d-8da4-749cb2b9ac5d,bronze_to_silver-20261011T130937Z-7f0a33,bronze_to_silver,Bronze-to-Silver,backfill,mixed,deleted_questions -> answers,,"batch_ids=*; dates=2026-09-25..2026-10-08 -> [full_2025-04-01_2025-07-01, incr_20260925T155859Z, full_2025-01-01_2025-04-01, incr_20261007T155914Z